In [24]:
import serial 
import serial.tools.list_ports

import time
import os 
import json
from pathlib import Path
from typing import Any, Dict, Optional


In [17]:
"""
action

A string that can be used to indicate intent.

In the current BP, it’s only used for one thing: if action is "stop", "quit", "exit" (case-insensitive), the BP will terminate.

Any other value (e.g. "measure") is effectively ignored and the BP proceeds normally.

co2_r

Sets the reference CO₂ setpoint (ppm / µmol mol⁻¹) via:

SETCONTROL("CO2_r", sp_co2, "float")

If wait_for_co2 is true, the BP will wait until measured CO2_r is close to this value (within co2_tol).

qin

Sets the incident light (PPFD) setpoint via:

SETCONTROL("Qin", sp_qin, "float")

Units are whatever your LI-6800 expects for Qin (typically µmol m⁻² s⁻¹ for PPFD).

flow

Sets the chamber flow setpoint via:

SETCONTROL("Flow", sp_flow, "float")

Units depend on your configuration (commonly µmol s⁻¹).

tair

Sets the chamber air temperature setpoint via:

SETCONTROL("Tair", sp_tair, "float")

In °C.

rh_air

Sets the chamber relative humidity setpoint via:

SETCONTROL("RH_air", sp_rh, "float")

In %RH.

fan_rpm

Sets the fan speed setpoint via:

SETCONTROL("Fan_rpm", sp_fan, "float")

In RPM.

pressure

Sets the chamber pressure (over-pressure) setpoint via:

SETCONTROL("Pressure", sp_p, "float")

Units depend on your configuration (often kPa or mbar equivalent in LI-COR’s control dictionary).

wait_for_co2

Boolean (true/false).

If true and co2_r was provided, the BP waits until:

abs(CO2_r_measured - co2_r_setpoint) < co2_tol

This is a stability/settling criterion for CO₂ before logging.

co2_tol

The tolerance used by wait_for_co2.

Default in the BP is 20 if you don’t provide it.

In ppm.

wait_s

Adds an extra fixed wait after setpoints (and after the optional CO₂ wait) using:

WAIT(dur=wait_s, units="Seconds")

Default in the BP is 10 seconds if not provided.

log

Boolean.

If true, the BP calls LOG() once, which writes one record to the currently open log file.

If no log file is open on the LI-6800, LOG() won’t produce a saved record.
"""

'\naction\n\nA string that can be used to indicate intent.\n\nIn the current BP, it’s only used for one thing: if action is "stop", "quit", "exit" (case-insensitive), the BP will terminate.\n\nAny other value (e.g. "measure") is effectively ignored and the BP proceeds normally.\n\nco2_r\n\nSets the reference CO₂ setpoint (ppm / µmol mol⁻¹) via:\n\nSETCONTROL("CO2_r", sp_co2, "float")\n\nIf wait_for_co2 is true, the BP will wait until measured CO2_r is close to this value (within co2_tol).\n\nqin\n\nSets the incident light (PPFD) setpoint via:\n\nSETCONTROL("Qin", sp_qin, "float")\n\nUnits are whatever your LI-6800 expects for Qin (typically µmol m⁻² s⁻¹ for PPFD).\n\nflow\n\nSets the chamber flow setpoint via:\n\nSETCONTROL("Flow", sp_flow, "float")\n\nUnits depend on your configuration (commonly µmol s⁻¹).\n\ntair\n\nSets the chamber air temperature setpoint via:\n\nSETCONTROL("Tair", sp_tair, "float")\n\nIn °C.\n\nrh_air\n\nSets the chamber relative humidity setpoint via:\n\nSETCONTR

In [18]:
# ssh licor@[fe80::f01c:15ff:feb8:9639%6]

In [ ]:

LI6800_HOST = r"licor@[fe80::f01c:15ff:feb8:9639%6]"
REMOTE_PATH = "/home/licor/apps/dynamic/remote_cmd.json"

def send_remote_cmd(
    cmd: Dict[str, Any],
    local_json_path: Path = Path("remote_cmd.json"),
    scp_exe: str = "scp",
    identity_file: Optional[Path] = None,   # e.g. Path(r"C:\Users\JII-c\.ssh\id_ed25519")
    strict_host_key_checking: bool = True,
) -> None:
    """
    Serialize `cmd` to JSON and SCP it to the LI-6800 remote command path.
    Requires that SSH key auth works for LI6800_HOST.
    """
    # 1) Write JSON locally
    local_json_path.write_text(json.dumps(cmd, indent=2), encoding="utf-8")

    # 2) Build scp command
    scp_cmd = [scp_exe]

    if identity_file:
        scp_cmd += ["-i", str(identity_file)]

    # Keep SSH from hanging on first-connect prompts (optional)
    if not strict_host_key_checking:
        scp_cmd += [
            "-o", "StrictHostKeyChecking=no",
            "-o", "UserKnownHostsFile=NUL",  # Windows null device
        ]

    # Increase verbosity if you want debugging: add "-v"
    scp_cmd += [str(local_json_path), f"{LI6800_HOST}:{REMOTE_PATH}"]

    # 3) Run
    result = subprocess.run(
        scp_cmd,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "scp failed\n"
            f"cmd: {' '.join(scp_cmd)}\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}\n"
        )

In [99]:
import json
import time
import uuid
import subprocess
from pathlib import Path
from typing import Any, Dict, Optional

LI6800 = r"licor@[fe80::f01c:15ff:feb8:9639%6]"
REMOTE_DIR = "/home/licor/apps/dynamic"
REMOTE_CMD = f"{REMOTE_DIR}/remote_cmd.json"
REMOTE_TMP = f"{REMOTE_DIR}/remote_cmd.json.tmp"
REMOTE_ACK = f"{REMOTE_DIR}/remote_ack.json"

def _run(cmd: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, capture_output=True, text=True)

def send_and_wait_ack(
    cmd: Dict[str, Any],
    *,
    local_cmd_path: Path = Path("remote_cmd.json"),
    local_ack_path: Path = Path("remote_ack.json"),
    identity_file: Optional[Path] = None,
    timeout_s: float = 120.0,
    poll_s: float = 0.5,
) -> Dict[str, Any]:
    """
    Sends a RemoteEnvMeasure command and blocks until the matching ack is written.

    Requires:
      - RemoteEnvMeasure.py writes remote_ack.json
      - RemoteEnvMeasure.py includes cmd_id in the ack (recommended)
    """

    # 1) Attach unique ID to command
    cmd_id = cmd.get("cmd_id") or str(uuid.uuid4())
    cmd = dict(cmd)
    cmd["cmd_id"] = cmd_id

    # 2) Write local JSON
    local_cmd_path.write_text(json.dumps(cmd, indent=2), encoding="utf-8")

    scp = ["scp"]
    ssh = ["ssh"]
    if identity_file:
        scp += ["-i", str(identity_file)]
        ssh += ["-i", str(identity_file)]

    # 3) Upload to tmp
    r1 = _run(scp + [str(local_cmd_path), f"{LI6800}:{REMOTE_TMP}"])
    if r1.returncode != 0:
        raise RuntimeError(f"SCP upload failed:\n{r1.stderr}")

    # 4) Atomic rename tmp -> cmd
    r2 = _run(ssh + [LI6800, f"mv {REMOTE_TMP} {REMOTE_CMD}"])
    if r2.returncode != 0:
        raise RuntimeError(f"SSH mv failed:\n{r2.stderr}")

    # 5) Poll until ack has our cmd_id
    t0 = time.time()
    last_err = ""
    while time.time() - t0 < timeout_s:
        # fetch ack (small file; simplest approach)
        r3 = _run(scp + [f"{LI6800}:{REMOTE_ACK}", str(local_ack_path)])
        if r3.returncode == 0:
            try:
                ack = json.loads(local_ack_path.read_text(encoding="utf-8"))
                if ack.get("cmd_id") == cmd_id:
                    return ack
            except Exception as e:
                last_err = f"ACK parse error: {e!r}"
        else:
            last_err = r3.stderr.strip()

        time.sleep(poll_s)

    raise TimeoutError(f"Timed out waiting for ack cmd_id={cmd_id}. Last error: {last_err}")


def send_serial_command(deviceport, command):
    with serial.Serial(deviceport, 115200, timeout=0.3) as ser:
        ser.write((command + "\r\n").encode())
        response = ser.readline().decode()
    return response
            



def write_output(ack, wiggly_data):
    try:
        with open("output.txt", "a", encoding="utf-8") as f:
            line = []
            line.append(json.dumps(ack))
            line.append(json.dumps(wiggly_data))
            f.write(",".join(line) + "\n")
            return True
    except Exception as e:
        print(f"Error writing output: {e!r}")
        return False


In [100]:
# List all available serial ports
ports = serial.tools.list_ports.comports()
print("Checking Serial Ports:")
for port in ports:
    with serial.Serial(port.device, 115200, timeout=2) as ser:
        ser.write("hello\r\n".encode())
        response =  ser.readline()
        print("Response:",response.decode())
        if "CO2 meter" in response.decode():
            deviceport = port.device
            print(f"Found CO2 sensor on {port.device}: {port.description}, Response: {response.decode().strip()}")



Checking Serial Ports:
Response: Hello CO2 meter ready

Found CO2 sensor on COM5: USB Serial Device (COM5), Response: Hello CO2 meter ready


In [105]:
send_serial_command(deviceport, "read_all")


'{"sample":{"as7341_data":{"f1_dark":0,"f2_dark":0,"f3_dark":0,"f4_dark":0,"c1_dark":0,"n1_dark":0,"f5_dark":0,"f6_dark":0,"f7_dark":0,"f8_dark":0,"c2_dark":0,"n2_dark":0,"f1_light":12030,"f2_light":12030,"f3_light":12030,"f4_light":12030,"c1_light":12030,"n1_light":12030,"f5_light":12030,"f6_light":12030,"f7_light":12030,"f8_light":12030,"c2_light":12030,"n2_light":12030,"f1":12030,"f2":12030,"f3":12030,"f4":12030,"c1":12030,"n1":12030,"f5":12030,"f6":12030,"f7":12030,"f8":12030,"c2":12030,"n2":12030},"BME680_data":{"temperature":20.57,"pressure":101358.08,"humidity":41.31,"gas_resistance":10392.98}}}\r\n'

In [ ]:
for temp in [20,25,30]:
    for rh in [40,45,50, 55, 60, 65 ,70]:
        for _ in range(3):
            j = 0
            for co2 in [200, 250, 300, 350 , 400, 450, 500, 450, 400, 350, 300, 250]:
                ack = send_and_wait_ack({
                    "action": "measure",
                    "co2_r": co2,
                    "tair": temp,
                    "rh_air": rh,
                    "wait_for_co2": True,
                    "co2_tol": 5,
                    "wait_s": 1,
                    "qin": 0,
                    "log": True
                })
                wiggly_data = send_serial_command(deviceport, "read_all")
                write_output(ack, wiggly_data)
                j += 1
                print(f"Completed measurement {j} for T={temp}°C, RH={rh}%, CO2={co2}ppm")


                time.sleep(5) # time between measurements

Completed measurement 1 for T=20°C, RH=40%, CO2=200ppm
Completed measurement 2 for T=20°C, RH=40%, CO2=250ppm
Completed measurement 3 for T=20°C, RH=40%, CO2=300ppm
Completed measurement 4 for T=20°C, RH=40%, CO2=350ppm
Completed measurement 5 for T=20°C, RH=40%, CO2=400ppm
Completed measurement 6 for T=20°C, RH=40%, CO2=450ppm
Completed measurement 7 for T=20°C, RH=40%, CO2=500ppm
Completed measurement 8 for T=20°C, RH=40%, CO2=450ppm
Completed measurement 9 for T=20°C, RH=40%, CO2=400ppm
Completed measurement 10 for T=20°C, RH=40%, CO2=350ppm
Completed measurement 11 for T=20°C, RH=40%, CO2=300ppm
Completed measurement 12 for T=20°C, RH=40%, CO2=250ppm
Completed measurement 1 for T=20°C, RH=40%, CO2=200ppm
Completed measurement 2 for T=20°C, RH=40%, CO2=250ppm
Completed measurement 3 for T=20°C, RH=40%, CO2=300ppm
Completed measurement 4 for T=20°C, RH=40%, CO2=350ppm
Completed measurement 5 for T=20°C, RH=40%, CO2=400ppm
Completed measurement 6 for T=20°C, RH=40%, CO2=450ppm
Complet

In [102]:

for temp in [25]:
    for rh in [60]:
        for _ in range(3):
            for co2 in [400]:
                ack = send_and_wait_ack({
                    "action": "measure",
                    "co2_r": co2,
                    "tair": temp,
                    "rh_air": rh,
                    "wait_for_co2": True,
                    "co2_tol": 5,
                    "wait_s": 1,
                    "qin": 0,
                    "log": True
                })
                wiggly_data = send_serial_command(deviceport, "read_all")
                write_output(ack, wiggly_data)

                time.sleep(5) # time between measurements

True

In [91]:
[wiggly_data, ack]

['{"sample":{"as7341_data":{"f1_dark":0,"f2_dark":0,"f3_dark":0,"f4_dark":5,"c1_dark":55,"n1_dark":0,"f5_dark":13,"f6_dark":20,"f7_dark":24,"f8_dark":15,"c2_dark":54,"n2_dark":0,"f1_light":12030,"f2_light":12030,"f3_light":12030,"f4_light":12030,"c1_light":12030,"n1_light":12030,"f5_light":12030,"f6_light":12030,"f7_light":12030,"f8_light":12030,"c2_light":12030,"n2_light":12030,"f1":12030,"f2":12030,"f3":12030,"f4":12025,"c1":11975,"n1":12030,"f5":12017,"f6":12010,"f7":12006,"f8":12015,"c2":11976,"n2":12030},"BME680_data":{"temperature":22.31,"pressure":101186.70,"humidity":41.31,"gas_resistance":6711.41}}}\r\n',
 {'ts': 1772193782.4864907,
  'cmd_id': '53dc96f7-119a-44c6-b9a1-f7ee41f3484e',
  'cmd': {'action': 'measure',
   'co2_r': 305,
   'tair': 24,
   'rh_air': 49,
   'wait_for_co2': True,
   'co2_tol': 20,
   'wait_s': 1,
   'qin': 0,
   'log': True,
   'cmd_id': '53dc96f7-119a-44c6-b9a1-f7ee41f3484e'},
  'meas': {'CO2_r': 305.814,
   'CO2_s': 305.438,
   'H2O_r': 16.406,
   'H2

In [ ]:
# ack = send_and_wait_ack({
#     "action": "measure",
#     "co2_r": 495,
#     "tair": 26,
#     "sp_rh": 49,
#     "wait_for_co2": True,
#     "co2_tol": 20,
#     "wait_s": 1,
#     "qin": 100,

#     "log": True
# })

In [ ]:
DEFAULT_CMD_PATH = Path("remote_cmd.json")
REMOTE_ACK = "/home/licor/apps/dynamic/remote_ack.json"
LI6800_HOST = r"licor@[fe80::f01c:15ff:feb8:9639%6]"

def write_remote_cmd_json(
    *,
    path: Path = DEFAULT_CMD_PATH,
    action: str = "measure",
    co2_r: Optional[float] = 400,
    qin: Optional[float] = 0,
    flow: Optional[float] = 500,
    tair: Optional[float] = 30,
    rh_air: Optional[float] = 60,
    fan_rpm: Optional[float] = 10000,
    pressure: Optional[float] = 0.2,
    wait_for_co2: Optional[bool] = False,
    co2_tol: Optional[float] = 20,
    wait_s: Optional[float] = 1,
    log: Optional[bool] = True,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Overwrite `path` with a RemoteEnvMeasure-compatible JSON command.

    Only parameters you provide (not None) are written (except `action`, which
    defaults to "measure"). Returns the dict that was written.

    Parameters match RemoteEnvMeasure.py keys:
      action, co2_r, qin, flow, tair, rh_air, fan_rpm, pressure,
      wait_for_co2, co2_tol, wait_s, log

    `extra` lets you add any additional keys (will overwrite existing keys).
    """
    cmd: Dict[str, Any] = {"action": action}

    def put(key: str, value: Any) -> None:
        if value is not None:
            cmd[key] = value

    put("co2_r", co2_r)
    put("qin", qin)
    put("flow", flow)
    put("tair", tair)
    put("rh_air", rh_air)
    put("fan_rpm", fan_rpm)
    put("pressure", pressure)
    put("wait_for_co2", wait_for_co2)
    put("co2_tol", co2_tol)
    put("wait_s", wait_s)
    put("log", log)

    if extra:
        cmd.update(extra)

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(cmd, indent=2), encoding="utf-8")
    return cmd

def fetch_remote_ack(
    *,
    local_path: Path = Path("remote_ack.json"),
    scp_exe: str = "scp",
    identity_file: Optional[Path] = None,
) -> Dict[str, Any]:
    scp_cmd = [scp_exe]
    if identity_file:
        scp_cmd += ["-i", str(identity_file)]

    scp_cmd += [f"{LI6800_HOST}:{REMOTE_ACK}", str(local_path)]

    res = subprocess.run(scp_cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"scp failed:\n{res.stderr}")

    return json.loads(local_path.read_text(encoding="utf-8"))

In [ ]:
# List all available serial ports
ports = serial.tools.list_ports.comports()
print("Checking Serial Ports:")
for port in ports:
    with serial.Serial(port.device, 115200, timeout=2) as ser:
        ser.write("hello\r\n".encode())
        response =  ser.readline()
        print("Response:",response.decode())
        if "CO2 meter" in response.decode():
            deviceport = port.device
            print(f"Found CO2 sensor on {port.device}: {port.description}, Response: {response.decode().strip()}")



Checking Serial Ports:
Response: Hello CO2 meter ready

Found CO2 sensor on COM5: USB Serial Device (COM5), Response: Hello CO2 meter ready


In [58]:
send_serial_command(deviceport, "read_all")

'{"sample":{"as7341_data":{"f1_dark":0,"f2_dark":9,"f3_dark":13,"f4_dark":46,"c1_dark":297,"n1_dark":0,"f5_dark":96,"f6_dark":159,"f7_dark":182,"f8_dark":103,"c2_dark":297,"n2_dark":0,"f1_light":12495,"f2_light":18000,"f3_light":18000,"f4_light":18000,"c1_light":18000,"n1_light":12649,"f5_light":18000,"f6_light":18000,"f7_light":18000,"f8_light":18000,"c2_light":18000,"n2_light":12783,"f1":12495,"f2":17991,"f3":17987,"f4":17954,"c1":17703,"n1":12649,"f5":17904,"f6":17841,"f7":17818,"f8":17897,"c2":17703,"n2":12783},"BME680_data":{"temperature":21.54,"pressure":101197.65,"humidity":48.54,"gas_resistance":281087.03}}}'

In [43]:
wiggly_data = []
licor_data = []
for i in [299,399]:
    cmd = write_remote_cmd_json(co2_r=i)
    send_remote_cmd(cmd)
    # wiggly_data.append(send_serial_command(deviceport, "read_all"))
    time.sleep(5)  # Give the LI-6800 a moment to process the command and update the ack file
    licor_data.append(fetch_remote_ack())

    time.sleep(5)

In [44]:
licor_data

[{'ts': 1772188658.8273423,
  'cmd': {'action': 'measure',
   'co2_r': 399,
   'qin': 0,
   'flow': 500,
   'tair': 30,
   'rh_air': 60,
   'fan_rpm': 10000,
   'pressure': 0.2,
   'wait_for_co2': True,
   'co2_tol': 20,
   'wait_s': 1,
   'log': True},
  'meas': {'CO2_r': 384.667,
   'CO2_s': 349.358,
   'H2O_r': 16.9001,
   'H2O_s': 13.9947,
   'Tchamber': 30.0111,
   'Tleaf': 29.6829,
   'RHcham': 33.28786121420776,
   'PPFD_in': 0}},
 {'ts': 1772188722.6539443,
  'cmd': {'action': 'measure',
   'co2_r': 299,
   'qin': 0,
   'flow': 500,
   'tair': 30,
   'rh_air': 60,
   'fan_rpm': 10000,
   'pressure': 0.2,
   'wait_for_co2': True,
   'co2_tol': 20,
   'wait_s': 1,
   'log': True},
  'meas': {'CO2_r': 314.125,
   'CO2_s': 362.928,
   'H2O_r': 23.8644,
   'H2O_s': 22.9973,
   'Tchamber': 29.9951,
   'Tleaf': 29.6677,
   'RHcham': 54.74849344086391,
   'PPFD_in': 0}}]

In [42]:
licor_data

[{'ts': 1772185354.255855,
  'cmd': {'action': 'measure',
   'co2_r': 399,
   'qin': 0,
   'flow': 500,
   'tair': 30,
   'rh_air': 20,
   'fan_rpm': 10000,
   'pressure': 0.2,
   'wait_for_co2': True,
   'co2_tol': 20,
   'wait_s': 1,
   'log': True},
  'meas': {'CO2_r': 385.132,
   'CO2_s': 349.026,
   'H2O_r': 8.29065,
   'H2O_s': 8.41602,
   'Tchamber': 30.004,
   'Tleaf': 29.6356,
   'RHcham': 20.025973938776627,
   'PPFD_in': 0}},
 {'ts': 1772188647.1673021,
  'cmd': {'action': 'measure',
   'co2_r': 299,
   'qin': 0,
   'flow': 500,
   'tair': 30,
   'rh_air': 60,
   'fan_rpm': 10000,
   'pressure': 0.2,
   'wait_for_co2': True,
   'co2_tol': 20,
   'wait_s': 1,
   'log': True},
  'meas': {'CO2_r': 313.041,
   'CO2_s': 361.535,
   'H2O_r': 13.8962,
   'H2O_s': 10.1678,
   'Tchamber': 30.0129,
   'Tleaf': 29.6864,
   'RHcham': 24.181292129318457,
   'PPFD_in': 0}}]